In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
class BVP:
    def __init__(self, patient, quadrant, signal, features, id):
        self.patient = patient
        self.quadrant = quadrant
        self.signal = signal
        self.features = features
        self.id = id

MAX_VID_LENGTH = 100

In [3]:
patients = list(range(1, 62))
patients.remove(23)

quadrants = [
    "Q1_1",
    "Q1_2",
    "Q2_1",
    "Q2_2",
    "Q3_1",
    "Q3_2",
    "Q4_1",
    "Q4_2",
    "Q5_1",
    "Q5_2",
    "Q6_1",
    "Q6_2",
    "Q7_1",
    "Q7_2",
    "Q8_1",
    "Q8_2",
    "Q9_1",
    "Q9_2"
]

In [4]:
bvp_path = "../BVPs"

failed_masks = [
    [2, "Q1_1"],
    [9, "Q7_2"],
    [12, "Q4_1"],
    [14, "Q1_2"],
    [52, "Q7_2"],
    [53, "Q4_2"],
    [60, "Q3_2"]
]

In [5]:
def cut_bvp(bvp, t_start, t_end, fs=60):
    n = len(bvp)

    # Convert time to sample indices
    n_start = int(t_start * fs)
    n_end = int(t_end * fs)

    # Clamp to valid range
    n_start = max(0, min(n_start, n))
    n_end = max(0, min(n_end, n))

    return bvp[n_start:n_end]

In [6]:
def cut_relevant_bvp(data, quadrant, fs=60):

    if quadrant == "Q1_2":

        data = cut_bvp(data, 20, MAX_VID_LENGTH, fs)

    elif quadrant == "Q3_2":

        data = cut_bvp(data, 38, MAX_VID_LENGTH, fs)
    
    return data


In [7]:
BVPs = []

fs = 60

for patient in patients:

    for quadrant in quadrants:

        if [patient, quadrant] in failed_masks:
            print(f"Skipping Patient_{patient}, {quadrant}")
            continue

        data = np.load(f"{bvp_path}/Patient_{patient}/{quadrant}.npy")

        data = cut_relevant_bvp(data, quadrant, fs)

        id = f"{patient}{quadrant}"

        bvp = BVP(patient, quadrant, data, [], id)

        BVPs.append(bvp)

        #print(f"Patient_{patient}, {quadrant}: {data.shape}")

print(f"Loaded {len(BVPs)} BVP signals")

Skipping Patient_2, Q1_1
Skipping Patient_9, Q7_2
Skipping Patient_12, Q4_1
Skipping Patient_14, Q1_2
Skipping Patient_52, Q7_2
Skipping Patient_53, Q4_2
Skipping Patient_60, Q3_2
Loaded 1073 BVP signals


In [8]:
from scipy.signal import find_peaks, czt

In [9]:
def compute_bvp_features_article(bvp_signal, fs, patient=None, quadrant=None):
    """
    Compute PPG features (14 features from paper) using CZT.
    Includes debug prints when something fails.
    """

    bvp = np.asarray(bvp_signal)

    if len(bvp) < fs * 2:
        print(f"[SHORT SIGNAL] Patient {patient}, {quadrant}")
        return None

    # ---- Remove DC ----
    bvp = bvp - np.mean(bvp)

    features = {}

    # --------------------------------------------------
    # 1. Peak detection → RR intervals
    # --------------------------------------------------
    min_dist = int(0.4 * fs)
    peaks, _ = find_peaks(bvp, distance=min_dist)

    if len(peaks) < 3:
        print(f"[PEAK DETECTION FAILED] Patient {patient}, {quadrant}")
        return None

    rr = np.diff(peaks) / fs
    hr = 60.0 / rr

    # --------------------------------------------------
    # 2. HR features
    # --------------------------------------------------
    features["hr_mean"] = np.mean(hr)
    features["hr_std"] = np.std(hr)

    # --------------------------------------------------
    # 3. HRV time-domain features
    # --------------------------------------------------
    features["rr_mean"] = np.mean(rr)
    features["rr_std"] = np.std(rr)

    diff_rr = np.diff(rr)

    if len(diff_rr) > 0:
        features["rmssd"] = np.sqrt(np.mean(diff_rr ** 2))
        features["sdsd"] = np.std(diff_rr)
        features["pnn50"] = np.sum(np.abs(diff_rr) > 0.05) / len(diff_rr)
    else:
        print(f"[RR DIFFERENCES FAILED] Patient {patient}, {quadrant}")
        features["rmssd"] = np.nan
        features["sdsd"] = np.nan
        features["pnn50"] = np.nan

    # --------------------------------------------------
    # 4. TINN
    # --------------------------------------------------
    try:
        hist, bin_edges = np.histogram(rr, bins=20)
        features["tinn"] = bin_edges[-1] - bin_edges[0]
    except Exception:
        print(f"[TINN FAILED] Patient {patient}, {quadrant}")
        features["tinn"] = np.nan

    # --------------------------------------------------
    # 5. Poincaré
    # --------------------------------------------------
    try:
        if len(diff_rr) > 0:
            sd1 = np.sqrt(np.var(diff_rr) / 2)
            sd2 = np.sqrt(2 * np.var(rr) - 0.5 * np.var(diff_rr))

            features["sd1"] = sd1
            features["sd2"] = sd2
            features["sd1_sd2"] = sd1 / sd2 if sd2 != 0 else np.nan
        else:
            raise ValueError
    except Exception:
        print(f"[POINCARE FAILED] Patient {patient}, {quadrant}")
        features["sd1"] = np.nan
        features["sd2"] = np.nan
        features["sd1_sd2"] = np.nan

    # --------------------------------------------------
    # 6. LF / HF using CZT
    # --------------------------------------------------
    try:
        rr_times = np.cumsum(rr)
        rr_times = np.insert(rr_times, 0, 0)

        fs_interp = 4.0
        t_uniform = np.arange(0, rr_times[-1], 1 / fs_interp)

        rr_interp = np.interp(
            t_uniform,
            rr_times,
            np.append(rr, rr[-1])
        )

        fmin = 0.04
        fmax = 0.4
        n_bins = 256

        w = np.exp(-1j * 2 * np.pi * (fmax - fmin) / (n_bins * fs_interp))
        a = np.exp(1j * 2 * np.pi * fmin / fs_interp)

        spectrum = czt(rr_interp, n_bins, w, a)
        power = np.abs(spectrum) ** 2
        freqs = np.linspace(fmin, fmax, n_bins)

        lf_band = (freqs >= 0.04) & (freqs < 0.15)
        hf_band = (freqs >= 0.15) & (freqs < 0.4)

        lf = np.sum(power[lf_band])
        hf = np.sum(power[hf_band])

        features["lf"] = lf
        features["hf"] = hf
        features["lf_hf"] = lf / hf if hf > 0 else np.nan

    except Exception:
        print(f"[LF/HF FAILED] Patient {patient}, {quadrant}")
        features["lf"] = np.nan
        features["hf"] = np.nan
        features["lf_hf"] = np.nan

    return features

In [10]:
bvp = BVPs[2]
fs = 60

features = compute_bvp_features_article(bvp.signal, fs, bvp.patient, bvp.quadrant)

In [11]:
print(bvp.patient, bvp.quadrant)
print(features)

1 Q2_1
{'hr_mean': 100.53924015095781, 'hr_std': 34.28763822791599, 'rr_mean': 0.6719298245614036, 'rr_std': 0.22313026236660477, 'rmssd': 0.27916222455598755, 'sdsd': 0.27909225198677157, 'pnn50': 0.6607142857142857, 'tinn': 0.5666666666666667, 'sd1': 0.19734802395647086, 'sd2': 0.2462275074155119, 'sd1_sd2': 0.8014865033882819, 'lf': 7305.553035238854, 'hf': 5825.711057788672, 'lf_hf': 1.254019116768881}


In [12]:
valid = []
failed = []

for bvp in BVPs:  # use a copy to safely remove items

    print(f"Computing Features for Patient_{bvp.patient}...", end="\r", flush=True)
    
    try:
        # Use the new windowed feature extraction
        feats = compute_bvp_features_article(bvp.signal, fs, bvp.patient, bvp.quadrant)

        if feats is None or feats == []:
            print(f"Failed: Patient_{bvp.patient}, {bvp.quadrant}")
            failed.append(f"Patient_{bvp.patient}, {bvp.quadrant}")
            BVPs.remove(bvp)  # remove problematic signal
        else:
            bvp.features = feats
            valid.append(f"Patient_{bvp.patient}, {bvp.quadrant}")

    except Exception as e:
        print(f"Error for Patient_{bvp.patient}, {bvp.quadrant}: {e}")
        failed.append(f"Patient_{bvp.patient}, {bvp.quadrant}")
        BVPs.remove(bvp)

print(f"Extracted features for {len(valid)} videos")
print(f"Failed: {failed}")

Extracted features for 1073 videos..
Failed: []


In [13]:
print(failed)

[]


In [14]:
for bvp in BVPs:

    try:

        if (f"Patient_{bvp.patient}, {bvp.quadrant}") not in failed:
            feats = bvp.features
            #print(bvp.patient, bvp.quadrant)
            for jdx in range(0, 14):
                dummie =  list(feats.values())[jdx] == np.nan

    except Exception as e:

        print(f"Not loaded on Patient_{bvp.patient}, {bvp.quadrant}")


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_selection import SelectFpr, f_classif

In [ ]:
# AROUSAL

def get_label_arousal(path):
    q = path.split("_")[0]

    if q in ["Q1", "Q2", "Q3"]:
        return "High Arousal"
    
    if q in ["Q4", "Q5", "Q6"]:
        return "Neutral Arousal"
    
    else:
        return "Low Arousal"
    
# VALENCE

def get_label_valence(path):
    q = path.split("_")[0]

    if q in ["Q1", "Q4", "Q7"]:
        return "Negative Valence"
    
    if q in ["Q2", "Q5", "Q8"]:
        return "Neutral Valence"
    
    else:
        return "Positive Valence"

In [17]:
X = []
y = []
groups = []

for bvp in BVPs:
    if bvp is None or bvp.features == []:
        print("Error: Patient", bvp.patient, bvp.quadrant)
        continue
    
    feat_values = list(bvp.features.values())
    X.append(feat_values)
    y.append(get_label(bvp.quadrant))
    groups.append(bvp.id)  
    

X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)
print(np.unique(y, return_counts=True))
print("Example of data: ", X[0])

(1073, 14) (1073,)
(array(['High Arousal', 'Low Arousal', 'Neutral Arousal'], dtype='<U15'), array([357, 358, 358], dtype=int64))
Example of data:  [9.26717515e+01 2.87402265e+01 7.03645833e-01 1.83746427e-01
 2.14400891e-01 2.14333471e-01 7.41935484e-01 5.66666667e-01
 1.51556651e-01 2.11083113e-01 7.17995147e-01 5.94819513e+03
 3.30514666e+03 1.79967661e+00]


In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [19]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [20]:
import tensorflow as tf

layers = tf.keras.layers
models = tf.keras.models

model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(3, activation='softmax')  # ✅ multiclass
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

c:\Users\mique\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [21]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    print(f"Fold {fold}")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Fit scaler ONLY on train
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = tf.keras.models.clone_model(model)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=30,
        batch_size=32,
        verbose=1
    )

Fold 0
Epoch 1/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.3430 - loss: 1.6430 - val_accuracy: 0.3953 - val_loss: 1.1077
Epoch 2/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3607 - loss: 1.4673 - val_accuracy: 0.3628 - val_loss: 1.0946
Epoch 3/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3387 - loss: 1.3918 - val_accuracy: 0.3721 - val_loss: 1.0896
Epoch 4/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3935 - loss: 1.2281 - val_accuracy: 0.3907 - val_loss: 1.0758
Epoch 5/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4062 - loss: 1.1985 - val_accuracy: 0.3860 - val_loss: 1.0833
Epoch 6/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3373 - loss: 1.2650 - val_accuracy: 0.4000 - val_loss: 1.0766
Epoch 7/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3987 - loss: 1.2277 - val_accuracy: 0.3814 - val_loss: 1.0688
Epoch 8/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4013 - loss: 1.1752 - val_accuracy: 0.4047 - v

In [22]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix


y_pred = model.predict(X_test)
y_pred = y_pred.argmax(axis=1)

print(classification_report(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
              precision    recall  f1-score   support

           0       0.45      0.72      0.55        71
           1       0.41      0.43      0.42        70
           2       0.44      0.16      0.24        73

    accuracy                           0.43       214
   macro avg       0.43      0.44      0.40       214
weighted avg       0.43      0.43      0.40       214

[[51 14  6]
 [31 30  9]
 [32 29 12]]
